# Tiny Dreamer Highway — Final Run: Iter 17 V2 (DreamerV2)

**Name:** Esteban Montelongo | **Course:** CSC 580 AI 2 | **Assignment:** Final Project — Dream the Road | **AI tools:** GitHub Copilot

---

This notebook trains and evaluates the **Iter 17 V2** DreamerV2 agent — the same environment and training schedule as the V1 score-optimized configuration, upgraded with DreamerV2 architectural improvements:

| V2 Change | What | Why |
|-----------|------|-----|
| **Categorical latents** | 32 categoricals × 32 classes = 1024-dim stochastic state | Multimodal representations — can commit to discrete world states instead of smearing across a Gaussian |
| **KL balancing** | α=0.8 dynamics / 0.2 representation with stop-gradients | Prevents posterior collapse by putting 4× more pressure on the prior to improve |
| **MSE decoder loss** | Direct MSE replaces Gaussian NLL | Balanced loss magnitudes — no more 11k constant dominating KL/reward terms |

## Experiment Design

| Phase | Environment | Config | Purpose |
|-------|-------------|--------|--------|
| **A — training** | 4 lanes, 12 cars | `final_run_iter17_v2.yaml` | Learn a speed-maximising continuous-control policy with V2 world model |
| **B — showcase** | 6 lanes, 45 cars | `showcase_iter17_v2.yaml` | Evaluate trained weights in denser, wider traffic |

**Sections:** Setup → Config → Training → Training curves → N-step prediction eval → Driving demos → Submission bundle

## 1 · Environment Setup

Mount Drive for artifact persistence (Colab VMs are ephemeral — everything under `/content/` is lost on disconnect), then clone/update the repo and install the package in editable mode so imports resolve from source.

In [1]:
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')
REPO_URL = 'https://github.com/estmon8u/CSC_580_Final_Project.git'
DRIVE_ROOT = Path('/content/drive/MyDrive/CSC_580_Final_Project')
ARTIFACT_ROOT = DRIVE_ROOT / 'artifacts'

for path in [DRIVE_ROOT, ARTIFACT_ROOT, ARTIFACT_ROOT / 'training_runs']:
    path.mkdir(parents=True, exist_ok=True)

print('Drive root:', DRIVE_ROOT)
print('Artifact root:', ARTIFACT_ROOT)

In [2]:
%%bash
set -e
REPO_URL='https://github.com/estmon8u/CSC_580_Final_Project.git'
BRANCH='Testing'
if [ ! -d /content/CSC_580_Final_Project/.git ]; then
  git clone -b "${BRANCH}" "${REPO_URL}" /content/CSC_580_Final_Project
else
  cd /content/CSC_580_Final_Project
  git fetch origin "${BRANCH}"
  git checkout "${BRANCH}"
  git pull --ff-only origin "${BRANCH}"
fi
cd /content/CSC_580_Final_Project
python -m pip install --upgrade pip --quiet
python -m pip install -e . --quiet

## 2 · Experiment Configuration

Same environment and training schedule as V1 Iter 17, with V2 model changes:

| Setting | Value | Why |
|---------|-------|-----|
| `frame_stack` | 3 | Gives the CNN instant velocity perception from pixel differences between stacked frames |
| `vehicles_count` | 12 | Sparser traffic so the agent can learn overtaking without constant collisions during warm-start |
| `npc_speed_scale` | 0.65 | Slower NPC traffic creates more overtaking opportunities |
| `num_categoricals` | 32 | **V2** — 32 independent categorical distributions for multimodal latent states |
| `num_classes` | 16 | **V2** — 16 classes per categorical → 512-dim stochastic state (reduced from 32 to cut decoder sparsity) |
| `kl_balance` | 0.8 | **V2** — 80% dynamics loss (train prior) / 20% representation loss (regularize encoder) |
| `steering_penalty` | 0.025 | Eliminates L/R jitter while still allowing deliberate lane changes |
| `batch_size` | 384 | Larger batches sample more diverse transitions |
| `imagination_horizon` | 20 | Long enough for multi-step credit assignment |
| `use_amp` / `bfloat16` | true | H100 tensor-core throughput |

All values come from `final_run_iter17_v2.yaml` — no in-memory overrides needed.

In [3]:
import json
import sys
import torch
from pathlib import Path

PROJECT_ROOT = Path('/content/CSC_580_Final_Project')
if str(PROJECT_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from tiny_dreamer_highway.config import load_experiment_config
from tiny_dreamer_highway.training import run_training_experiment

CONFIG_PATH = PROJECT_ROOT / 'notebooks' / 'configs' / 'final_run_iter17_v2.yaml'
config = load_experiment_config(CONFIG_PATH)

print(json.dumps(config.model_dump(), indent=2))

gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none'
print('Loaded config from:', CONFIG_PATH)
print('GPU:', gpu_name)
print(f'Stochastic dim: {config.model.num_categoricals} × {config.model.num_classes} = {config.model.stochastic_dim}')
print(f'KL balance: {config.training.kl_balance}')

## 3 · Training

`run_training_experiment` drives the outer loop. Each *cycle* runs:
1. **World model update** (`world_model_updates_per_cycle` gradient steps) — sample sequences from the replay buffer, compute reconstruction MSE + KL (balanced dynamics/representation) + reward + continue losses, update encoder/RSSM/decoder weights.
2. **Behavior update** (`behavior_updates_per_cycle` gradient steps) — imagine `imagination_horizon` steps via the RSSM prior, build λ-return targets, update actor and critic.
3. **Environment collection** (`policy_steps` real env steps) — run the trained actor, add new transitions to the replay buffer.

`warm_start_steps` pre-fills the replay buffer with random transitions before any gradient updates begin. `RESUME_FROM = None` starts fresh; set it to a checkpoint path to continue an interrupted run.

In [ ]:
RUN_NAME = 'final_iter17_v2'
RUN_ARTIFACT_ROOT = ARTIFACT_ROOT / 'training_runs' / RUN_NAME
RUN_ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)

# All values come from the YAML — no overrides needed.
CYCLES              = 500   # YAML: 2000
WARM_START_STEPS    = None   # YAML: 10000
POLICY_STEPS        = None
CHECKPOINT_INTERVAL = None

RESUME_FROM = None  # Fresh V2 run — V1 checkpoints are incompatible

print('Run name:', RUN_NAME)
print('Artifact root:', RUN_ARTIFACT_ROOT)
print('Resume from:', RESUME_FROM if RESUME_FROM else '(fresh run)')
print('Effective cycles:', config.training.cycles if CYCLES is None else CYCLES)
print('Effective warm-start steps:',
      config.training.warm_start_steps if WARM_START_STEPS is None else WARM_START_STEPS)
print('Sequence length:', config.replay.sequence_length)
print('Replay capacity:', config.replay.capacity)

### Launch

Override any YAML default by setting the variable to a non-`None` value. The training loop prints one line per cycle with current losses and evaluation reward.

In [5]:
print('Launching Iter 17 V2 final run (DreamerV2).')
print('Environment: 4 lanes, 12 cars — iter 17 config with V2 world model.')
print('Per-cycle progress lines will appear below.\n')

training_summary = run_training_experiment(
    config,
    RUN_ARTIFACT_ROOT,
    cycles=CYCLES,
    warm_start_steps=WARM_START_STEPS,
    policy_steps=POLICY_STEPS,
    checkpoint_interval=CHECKPOINT_INTERVAL,
    resume_from=RESUME_FROM,
)

print('\nCompleted cycles:', training_summary.completed_cycles)
print('Latest checkpoint:', training_summary.latest_checkpoint)
print('Latest metrics:', training_summary.latest_record)

## 4 · Training History

`export_training_history_artifacts` reads `cycle_metrics.csv` and writes a multi-panel PNG (`curves`) and a JSON summary (`summary`).

**What to look for in the V2 curves:**
- `reconstruction_loss` — now MSE (range 0–1), should decrease steadily. Much smaller than V1's Gaussian NLL (~11k).
- `kl_loss` — balanced dynamics + representation terms. Should settle at a stable positive value.
- `kl_dynamics` / `kl_representation` — V2 tracks these separately. Dynamics loss (80% weight) should be larger.
- `evaluation/mean_reward` — the primary performance signal; should trend upward as the actor learns.
- `evaluation/crash_rate` — should decrease toward zero for a robust policy.

In [6]:
from IPython.display import Image, display
import importlib
import json
import tiny_dreamer_highway.evaluation.training_analysis as _ta_mod

_ta_mod = importlib.reload(_ta_mod)
export_training_history_artifacts = _ta_mod.export_training_history_artifacts

history_artifacts = export_training_history_artifacts(
    training_summary.log_dir / 'cycle_metrics.csv',
    RUN_ARTIFACT_ROOT / 'analysis',
    prefix=RUN_NAME,
)

# export_training_history_artifacts returns:
#   history_artifacts['curves']  — multi-panel learning-curves PNG
#   history_artifacts['summary'] — JSON Path with per-metric summary stats
display(Image(filename=str(history_artifacts['curves'])))
analysis_summary = json.loads(history_artifacts['summary'].read_text(encoding='utf-8'))
print(json.dumps(analysis_summary, indent=2))

## 5 · N-step World Model Prediction Evaluation

Given a **seed observation** and the **real actions** recorded in the replay buffer, the world model is asked to predict what the next `EVAL_HORIZON` frames would look like using only the RSSM prior — no observations after the first.

**Protocol:**
1. Encode the seed frame → posterior state (RSSM grounded in reality).
2. Roll forward `EVAL_HORIZON` steps via the prior, applying the recorded actions.
3. Decode each imagined latent to a pixel prediction.
4. Compare predicted vs. actual frames with MSE, PSNR, and SSIM per step.

**V2 note:** The categorical latent space should produce sharper predictions than V1's Gaussian, especially for multi-step rollouts where the prior can commit to specific vehicle configurations rather than averaging across modes.

In [7]:
import torch
from tiny_dreamer_highway.data.replay_buffer import ReplayBuffer
from tiny_dreamer_highway.evaluation.policy_rollout import _load_models_from_checkpoint
from tiny_dreamer_highway.evaluation.prediction_eval import (
    evaluate_n_step_predictions,
    evaluate_latent_rollout_consistency,
)
from tiny_dreamer_highway.evaluation.visualization import export_prediction_media_bundle

# ── Load world model and actor weights from the latest checkpoint ────────────
DEVICE = torch.device(config.device if torch.cuda.is_available() else 'cpu')
CKPT_PATH = training_summary.latest_checkpoint
world_model, actor = _load_models_from_checkpoint(CKPT_PATH, config, DEVICE)
print('World model loaded from:', CKPT_PATH)

# ── Restore the replay buffer for ground-truth observation sequences ──────────
step_num = int(CKPT_PATH.stem.split('_')[-1])
replay_path = CKPT_PATH.with_name(f'replay_{step_num:05d}.pt')
replay_buffer = ReplayBuffer(capacity=config.replay.capacity)
replay_state = torch.load(replay_path, map_location='cpu', weights_only=False)
replay_buffer.load_state_dict(replay_state)
print(f'Replay buffer restored: {len(replay_buffer):,} transitions  '
      f'(capacity: {replay_buffer.capacity:,})')

In [8]:
# ── Sample ground-truth sequences from the replay buffer ─────────────────────
EVAL_HORIZON = 15   # Number of imagination steps
EVAL_BATCH   = 8    # Sequences per evaluation (metrics averaged over this)

seq_batch = replay_buffer.sample_sequence_batch(
    batch_size=EVAL_BATCH,
    sequence_length=EVAL_HORIZON + 1,
)

seed_obs       = torch.as_tensor(seq_batch.observations[:, 0], device=DEVICE)
future_actions = torch.as_tensor(
    seq_batch.actions[:, :EVAL_HORIZON], dtype=torch.float32, device=DEVICE
)
target_obs     = torch.as_tensor(seq_batch.observations[:, 1:EVAL_HORIZON + 1], device=DEVICE)

print(f'seed_obs:       {tuple(seed_obs.shape)}')
print(f'future_actions: {tuple(future_actions.shape)}')
print(f'target_obs:     {tuple(target_obs.shape)}')

# ── Pixel-level n-step prediction evaluation ─────────────────────────────────
n_step_result = evaluate_n_step_predictions(world_model, seed_obs, future_actions, target_obs)

print('\nN-step prediction summary (average over all steps and batch):')
for key, value in n_step_result['summary'].items():
    print(f'  {key}: {value:.4f}')

print('\nPer-step pixel metrics (first 5 steps):')
for item in n_step_result['step_metrics'][:5]:
    step = int(item['step'])
    print(f"  step {step:2d}  MSE={item['mse']:.4f}  "
          f"PSNR={item['psnr']:.2f} dB  SSIM={item['ssim']:.4f}")

### Prediction Media Export

`export_prediction_media_bundle` produces three files:
- **`metrics_plot.png`** — MSE / PSNR / SSIM curves vs. imagination step.
- **`comparison_grid.png`** — rows = steps, columns = Target | Predicted | Absolute Error (magma colormap).
- **`comparison_video.gif`** — animated version of the grid at 2 fps.

In [9]:
from IPython.display import Image, display

pred_media = export_prediction_media_bundle(
    n_step_result['step_metrics'],
    n_step_result['predictions'],
    target_obs,
    RUN_ARTIFACT_ROOT / 'world_model_predictions',
    prefix=RUN_NAME,
    max_steps=EVAL_HORIZON,
    fps=2,
)

print('Exported prediction artifacts:')
for key, path in pred_media.items():
    print(f'  {key}: {path.name}')

print('\nPer-step metric curves (MSE / PSNR / SSIM vs imagination step):')
display(Image(filename=str(pred_media['metrics_plot'])))

print('\nComparison grid — Target | Predicted | Absolute Error per step:')
display(Image(filename=str(pred_media['comparison_grid'])))

print('\nPrediction GIF — animated Target | Predicted | Error:')
display(Image(filename=str(pred_media['comparison_video'])))

### Latent Rollout Consistency

Measures how far the RSSM's **imagined** latent state drifts from the **observation-corrected posterior** state when the same actions are applied.

Two paths from the same starting state:
- **Grounded** — at every step the model receives the real observation and updates via the posterior.
- **Imagined** — the model runs entirely via the prior, no observations after step 0.

`latent_mse` is the feature-space MSE between these two paths at each step. With V2's KL balancing, the dynamics loss (80% weight) should keep the prior tightly aligned with the posterior, leading to slower drift than V1.

In [10]:
# ── Latent-space rollout consistency ─────────────────────────────────────────
latent_result = evaluate_latent_rollout_consistency(
    world_model,
    seed_obs,
    future_actions,
    target_obs,
)

print('=== Latent Rollout Consistency ===')
print(f"Summary: {latent_result['summary']}")
print()
print(f"{'Step':>4}  {'Det MSE':>12}  {'Stoch MSE':>12}  {'Feature MSE':>12}  {'Prior-Post KL':>14}")
print('-' * 62)
for s in latent_result['step_metrics']:
    print(
        f"{int(s['step']):>4}  "
        f"{s['deterministic_mse']:>12.6f}  "
        f"{s['stochastic_mse']:>12.6f}  "
        f"{s['feature_mse']:>12.6f}  "
        f"{s['prior_posterior_kl']:>14.6f}"
    )

## 6 · Agent Driving Demo — Training Environment (Phase A)

Record the trained actor in the **same 4-lane, 12-car environment it was trained in**. At inference the policy runs a real-time RSSM posterior loop: encode frame → observe step → actor mode (tanh of mean, not a sample) → smooth with previous action → scale and execute.

5 episodes, up to 400 steps each. A well-trained agent should accelerate in clear lanes and avoid collisions.

In [11]:
from tiny_dreamer_highway.evaluation.policy_rollout import record_demo_videos
from IPython.display import Image, display

demo_output_dir = RUN_ARTIFACT_ROOT / 'demo_training_env'

demo_bundle = record_demo_videos(
    config=config,
    checkpoint_path=CKPT_PATH,
    output_dir=demo_output_dir,
    num_episodes=5,
    max_steps=400,
    fps=15,
    seed=42,
    prefix=f'{RUN_NAME}_train_env',
    device=DEVICE,
)

ep_rewards = [result.total_reward for result in demo_bundle.results]
ep_lengths = [result.steps for result in demo_bundle.results]

print(f'Episodes recorded : {len(demo_bundle.video_paths)}')
print(f'Mean reward       : {sum(ep_rewards)/len(ep_rewards):.3f}')
print(f'Mean episode len  : {sum(ep_lengths)/len(ep_lengths):.1f}')
print(f'Videos saved to   : {demo_output_dir}')

if demo_bundle.video_paths:
    print('\nTraining-env demo GIFs:')
    for index, video_path in enumerate(demo_bundle.video_paths, start=1):
        print(f'  Episode {index}: {video_path.name}')
        display(Image(filename=str(video_path)))
else:
    print('No demo GIFs were written; check env.render() / render_mode configuration.')

## 7 · Showcase Demo — 6 Lanes, 45 Cars (Phase B)

Evaluate the **same checkpoint** in a harder environment not seen during training. No fine-tuning — the same weights are deployed directly.

5 episodes, up to 600 steps each. Some reward reduction compared to Phase A is expected due to denser traffic; qualitative behaviour (staying on road, overtaking) should remain intact.

In [12]:
from tiny_dreamer_highway.config import load_experiment_config

showcase_config = load_experiment_config(
    PROJECT_ROOT / 'notebooks' / 'configs' / 'showcase_iter17_v2.yaml'
)
print(f"Showcase env: {showcase_config.env.lanes_count} lanes, "
      f"{showcase_config.env.vehicles_count} vehicles")

showcase_output_dir = RUN_ARTIFACT_ROOT / 'demo_showcase_env'

showcase_bundle = record_demo_videos(
    config=showcase_config,
    checkpoint_path=CKPT_PATH,   # same weights as training-env demo
    output_dir=showcase_output_dir,
    num_episodes=5,
    max_steps=600,
    fps=15,
    seed=99,
    prefix=f'{RUN_NAME}_showcase',
    device=DEVICE,
)

sc_rewards = [result.total_reward for result in showcase_bundle.results]
sc_lengths = [result.steps for result in showcase_bundle.results]

print(f'\nShowcase Demo (6 lanes / 45 cars):')
print(f'  Episodes recorded : {len(showcase_bundle.video_paths)}')
print(f'  Mean reward       : {sum(sc_rewards)/len(sc_rewards):.3f}')
print(f'  Mean episode len  : {sum(sc_lengths)/len(sc_lengths):.1f}')
print(f'  Videos saved to   : {showcase_output_dir}')

if showcase_bundle.video_paths:
    print('\nShowcase demo GIFs:')
    for index, video_path in enumerate(showcase_bundle.video_paths, start=1):
        print(f'  Episode {index}: {video_path.name}')
        display(Image(filename=str(video_path)))
else:
    print('No showcase GIFs were written; check env.render() / render_mode configuration.')

## 8 · Submission Bundle

`export_submission_bundle` copies all artifact files into a single directory and writes a `manifest.json` with the bundle name, file list, and any metadata passed in. `create_archive=True` additionally tars and gzips the directory.

| Artifact | Description |
|----------|-------------|
| Training curves PNG | Multi-panel learning-curves figure |
| Checkpoint | World model + actor + critic weights |
| Comparison grid PNG | Target vs. predicted frames per step |
| Prediction metrics plot | MSE / PSNR / SSIM curves |
| Prediction GIF | Animated target / predicted / error |
| Demo videos | Phase A (training env) and Phase B (showcase) first episodes |
| JSON summaries | N-step summary, latent summary, episode stats |

In [13]:
from tiny_dreamer_highway.evaluation.artifact_bundle import export_submission_bundle
import json
import datetime

# ── Compute episode stats from DemoBundle.results ────────────────────────────
ep_rewards = [result.total_reward for result in demo_bundle.results]
sc_rewards = [result.total_reward for result in showcase_bundle.results]

# ── Write lightweight JSON sidecar summaries ──────────────────────────────────
summary_dir = RUN_ARTIFACT_ROOT / 'summaries'
summary_dir.mkdir(parents=True, exist_ok=True)

sidecars = {
    'n_step_summary.json': n_step_result['summary'],
    'latent_summary.json': latent_result['summary'],
    'demo_train_stats.json': {
        'num_episodes': len(demo_bundle.video_paths),
        'mean_reward': sum(ep_rewards) / len(ep_rewards),
        'rewards': ep_rewards,
    },
    'demo_showcase_stats.json': {
        'num_episodes': len(showcase_bundle.video_paths),
        'mean_reward': sum(sc_rewards) / len(sc_rewards),
        'rewards': sc_rewards,
    },
}
for fname, data in sidecars.items():
    (summary_dir / fname).write_text(json.dumps(data, indent=2, default=str))

# ── Bundle all artifacts ──────────────────────────────────────────────────────
artifacts = {
    'training_curves': history_artifacts['curves'],
    'checkpoint': CKPT_PATH,
    'prediction_metrics_plot': pred_media['metrics_plot'],
    'comparison_grid': pred_media['comparison_grid'],
    'prediction_gif': pred_media['comparison_video'],
    'n_step_summary': summary_dir / 'n_step_summary.json',
    'latent_summary': summary_dir / 'latent_summary.json',
    'demo_train_stats': summary_dir / 'demo_train_stats.json',
    'demo_showcase_stats': summary_dir / 'demo_showcase_stats.json',
}
if demo_bundle.video_paths:
    artifacts['demo_train_ep0'] = demo_bundle.video_paths[0]
if showcase_bundle.video_paths:
    artifacts['demo_showcase_ep0'] = showcase_bundle.video_paths[0]

bundle_paths = export_submission_bundle(
    artifacts=artifacts,
    output_dir=DRIVE_ROOT / 'submission',
    bundle_name=f'csc580_final_{RUN_NAME}',
    metadata={
        'run_name': RUN_NAME,
        'timestamp': datetime.datetime.now().isoformat(),
        'architecture': 'DreamerV2',
        'num_categoricals': config.model.num_categoricals,
        'num_classes': config.model.num_classes,
        'stochastic_dim': config.model.stochastic_dim,
        'kl_balance': config.training.kl_balance,
        'lanes': config.env.lanes_count,
        'vehicles': config.env.vehicles_count,
        'n_step_summary': n_step_result['summary'],
        'latent_summary': latent_result['summary'],
        'demo_mean_reward': sum(ep_rewards) / len(ep_rewards),
        'showcase_mean_reward': sum(sc_rewards) / len(sc_rewards),
    },
    create_archive=True,
)

print('=== Submission Bundle ===')
for key, path in bundle_paths.items():
    print(f'  {key}: {path}')
print('\nUpload the .tar.gz to the class submission portal.')